In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes --quiet

In [3]:
import torch

In [4]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

PyTorch version: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8


In [5]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit)

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


In [8]:
text = "I love using Unsloth in Colab!"

In [9]:
tokens = tokenizer.tokenize(text)
print(f"Tokens: {tokens}")

Tokens: ['I', 'Ġlove', 'Ġusing', 'ĠUns', 'loth', 'Ġin', 'ĠCol', 'ab', '!']


In [10]:
input_ids = tokenizer.encode(text)
print(f"IDs:    {input_ids}")

IDs:    [128000, 40, 3021, 1701, 62143, 70652, 304, 4349, 370, 0]


In [11]:
decoded = tokenizer.decode(input_ids)
print(f"Back:   '{decoded}'")

Back:   '<|begin_of_text|>I love using Unsloth in Colab!'


In [12]:
print(tokenizer.special_tokens_map)

{'bos_token': '<|begin_of_text|>', 'eos_token': '<|eot_id|>', 'pad_token': '<|finetune_right_pad_id|>'}


In [13]:
print(f"{'ID':<10} {'Token':<20}")
print("-" * 30)

# Loop through all special IDs and decode them
for id_num in tokenizer.all_special_ids:
    # We decode the ID to see the text representation
    token_text = tokenizer.decode([id_num])
    print(f"{id_num:<10} {token_text:<20}")


ID         Token               
------------------------------
128000     <|begin_of_text|>   
128009     <|eot_id|>          
128004     <|finetune_right_pad_id|>


In [14]:
# Check specific IDs that we know belong to Llama-3 structure tags
interesting_ids = [128001, 128002, 128003, 128004, 128005, 128006, 128007, 128008, 128009, 128010]

print(f"{'ID':<10} {'Token String':<30}")
print("-" * 40)

for id_num in interesting_ids:
    # Decode strictly to see the text
    token_str = tokenizer.decode([id_num])
    print(f"{id_num:<10} {repr(token_str):<30}")


ID         Token String                  
----------------------------------------
128001     '<|end_of_text|>'             
128002     '<|reserved_special_token_0|>'
128003     '<|reserved_special_token_1|>'
128004     '<|finetune_right_pad_id|>'   
128005     '<|reserved_special_token_2|>'
128006     '<|start_header_id|>'         
128007     '<|end_header_id|>'           
128008     '<|eom_id|>'                  
128009     '<|eot_id|>'                  
128010     '<|python_tag|>'              


In [15]:
print(tokenizer.chat_template)

{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{{- "Cutting Knowledge Date: December 2023\n" }}
{{- 

In [16]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "List 3 famous distinct colors of apples."}
]

In [17]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt").to("cuda")

In [18]:
inputs

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1032,  17907,    220,   2366,     21,    271,   2675,    527,
            264,  11190,  18328,     13, 128009, 128006,    882, 128007,    271,
            861,    220,     18,  11495,  12742,   8146,    315,  41776,     13,
         128009, 128006,  78191, 128007,    271]], device='cuda:0')

In [19]:
output = model.generate(
    input_ids = inputs,
    max_new_tokens = 128,
    use_cache = True,
    temperature = 1.5,
    min_p = 0.1
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [20]:
output

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1032,  17907,    220,   2366,     21,    271,   2675,    527,
            264,  11190,  18328,     13, 128009, 128006,    882, 128007,    271,
            861,    220,     18,  11495,  12742,   8146,    315,  41776,     13,
         128009, 128006,  78191, 128007,    271,   8586,    527,   2380,  11495,
          12742,   8146,    315,  41776,   1473,     16,     13,   3146,   6161,
          96618,   3861,    315,    279,   1455,  66352,  24149,   8146,     11,
           3816,  41776,    527,   3967,    369,    872,  10437,    323,  10284,
          45915,  17615,    627,     17,     13,   3146,  60890,  96618,  18288,
          41776,    527,   3967,    369,    872,  10107,  14071,   1933,    323,
          10437,     11,  23900,  17615,     13,   2435,    527,   3629,   1511,
            304,  24149,  77

In [21]:
decoded_output = tokenizer.batch_decode(output)

In [22]:
decoded_output

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 13 Sep 2026\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nList 3 famous distinct colors of apples.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHere are three famous distinct colors of apples:\n\n1. **Red**: One of the most recognizable apple colors, Red apples are known for their sweet and slightly tart flavor.\n2. **Golden**: Golden apples are known for their bright yellow color and sweet, mild flavor. They are often used in apple salads and snacks.\n3. **Green**: Green apples are a popular choice for baking, cooking, and making apple products like juice and sauce. They are often tart and crisp, with a hint of sweetness.<|eot_id|>']

In [23]:
decoded_output[0]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 13 Sep 2026\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nList 3 famous distinct colors of apples.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHere are three famous distinct colors of apples:\n\n1. **Red**: One of the most recognizable apple colors, Red apples are known for their sweet and slightly tart flavor.\n2. **Golden**: Golden apples are known for their bright yellow color and sweet, mild flavor. They are often used in apple salads and snacks.\n3. **Green**: Green apples are a popular choice for baking, cooking, and making apple products like juice and sauce. They are often tart and crisp, with a hint of sweetness.<|eot_id|>'

In [24]:
print(decoded_output[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 13 Sep 2026

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

List 3 famous distinct colors of apples.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here are three famous distinct colors of apples:

1. **Red**: One of the most recognizable apple colors, Red apples are known for their sweet and slightly tart flavor.
2. **Golden**: Golden apples are known for their bright yellow color and sweet, mild flavor. They are often used in apple salads and snacks.
3. **Green**: Green apples are a popular choice for baking, cooking, and making apple products like juice and sauce. They are often tart and crisp, with a hint of sweetness.<|eot_id|>


In [25]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
    )

Unsloth 2026.9.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [26]:
import json

file_path = "/content/support.jsonl"

raw_data = []

with open(file_path, 'r', encoding='utf-8') as f:
  for line in f:
    raw_data.append(json.loads(line))

print(raw_data[0])

{'input': 'I want to know the pricing for your enterprise plan', 'output': 'SALES_TEAM'}


In [27]:
from datasets import Dataset

In [28]:
dataset = Dataset.from_list(raw_data)

In [29]:
type(dataset)

datasets.arrow_dataset.Dataset

In [30]:
dataset[1]

{'input': 'Can you share details about bulk licensing discounts',
 'output': 'SALES_TEAM'}

In [31]:
# System Prompt
# User Prompt
# Assistant Prompt

In [32]:
def formatting_prompts_func(examples):
    convos = []
    texts = []

    system_prompt = "You are an intelligent routing assistant. Classify the user query into the correct department. Output ONLY the label (e.g., SALES_TEAM"

    for input_text, output_text in zip(examples["input"], examples["output"]):
        conversation = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": input_text},
            {"role": "assistant", "content": output_text},
        ]

        text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        texts.append(text)

    return {"text": texts}

In [33]:
dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/199 [00:00<?, ? examples/s]

In [34]:
dataset[0]

{'input': 'I want to know the pricing for your enterprise plan',
 'output': 'SALES_TEAM',
 'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 13 Sep 2026\n\nYou are an intelligent routing assistant. Classify the user query into the correct department. Output ONLY the label (e.g., SALES_TEAM<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nI want to know the pricing for your enterprise plan<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nSALES_TEAM<|eot_id|>'}

In [35]:
# !pip install -U trl

import transformers
import trl
import unsloth
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

In [36]:
print(f"Transformers: ", transformers.__version__)
print(f"Unsloth:      ", unsloth.__version__)
print(f"trl:         ", trl.__version__)

Transformers:  5.5.0
Unsloth:       2026.9.4
trl:          0.8.6


In [37]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

Map (num_proc=2):   0%|          | 0/199 [00:00<?, ? examples/s]

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [38]:
trainer_stats = trainer.train()

NameError: name 'trainer' is not defined

In [ ]:
# Inference step

FastLanguageModel.for_inference(model)

In [ ]:
new_input = "We are buying license for all the team!"
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": new_input}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True, temperature = .1)
decoded_output = tokenizer.batch_decode(outputs)

print(decoded_output[0])

In [ ]:
# Evalution step - running with multiple test data

In [ ]:
# pushing fine tuned model to HuggingFaces
from huggingface_hub import login
login("hf_**")

model.push_to_hub_merged("audhil/llama-3.2-3B-Instruct-bnb-4bit-classification-merged", tokenizer, save_method = 'merged_16bit')